# Run X6Y3 experiments with LeeQ

Use the **LeeQ — Huracan experiments** kernel. This notebook connects through the existing Huracan RPC tunnel and runs native LeeQ sequences on Q0 or Q1. The original notebook calibration remains the baseline.

1. Run setup and choose the experiment below.
2. Leave `RUN_HARDWARE=False` to compile and inspect the plan without taking shots.
3. When you intend to use the board, set `RUN_HARDWARE=True`, rerun the parameter cell, then run the acquisition cell.
4. Inspect the plots and fit report. Rerunning acquisition creates a fresh output directory. Use **Kernel → Interrupt Kernel** to stop between submissions; completed point data is preserved. An in-flight shot batch can finish after interruption.

Choose `readout`, `ramsey`, `ramsey_three_stage`, or `pingpong`. Start with a fresh readout if the old reference no longer matches current conditions. `ramsey_three_stage` follows the LeeQ example and stops if a fit is rejected. For the paused session's saved results, open [X6Y3_TuneUp_Inspection.ipynb](X6Y3_TuneUp_Inspection.ipynb).

Opening or running the default notebook does not take shots. No firmware load, remote installation, service restart or persistent RF reconfiguration is performed. Calibration updates are saved separately only when you choose to export reviewed settings.

In [2]:
import os, sys, json
from pathlib import Path
from datetime import datetime, timezone
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
LEEQ_ROOT = Path('/local/data/projects/LeeQ-lbnl-qubic')
HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
sys.path.insert(0, str(LEEQ_ROOT))
sys.path.append(str(HARNESS_ROOT / 'artifacts/x6y3/venv/lib/python3.11/site-packages'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from leeq.setups.x6y3 import X6Y3Calibration
from leeq.setups.huracan import create_huracan_setup
from leeq.experiments.experiments import ExperimentManager
from leeq.experiments.x6y3 import acquire_plan, readout_plan, analyze_readout
from leeq.experiments.x6y3_tuneup import (
    RAMSEY_STAGES, leeq_ramsey_plan, pingpong_plan, fit_ramsey, fit_pingpong, save_candidate,
)
from leeq.experiments.x6y3_validation import verify_plan_parity

calibration = X6Y3Calibration(HARNESS_ROOT / 'X6Y3/configs_new/config.yaml')
configured = create_huracan_setup('/local/data/artifacts/zcu216/20260903T_aa01c78f_rdrv230_d4_candidate/bits_aa01c78f/channel_config.json')
manager = ExperimentManager()
if configured.name in manager.get_available_setup_names():
    huracan = manager.get_setup(configured.name)
    assert huracan.rpc_uri == configured.rpc_uri and huracan.channel_config_sha256 == configured.channel_config_sha256
else:
    huracan = configured
_readout_references = globals().get('_readout_references', {})
print('LeeQ:', LEEQ_ROOT)
print('RPC endpoint:', huracan.rpc_uri, '(not contacted yet)')
print('Baseline SHA256:', calibration.sha256)

LeeQ: /local/data/projects/LeeQ-lbnl-qubic
RPC endpoint: http://127.0.0.1:9095 (not contacted yet)
Baseline SHA256: dcabca385b95661a9eba44ee99315a27da9c65b62db44c4ef523f0cd1a2986c8


## Experiment controls

All frequency corrections are relative to the immutable original YAML, in MHz. The paused session's provisional centers were Q0 `-0.2047612346709513` and Q1 `0.03252838075279792`; these are not independently validated. Amplitude scales multiply the original pulse amplitudes.

For ping-pong, X90 counts must be multiples of four; X counts must be even. Tune the independent X90 and X definitions separately. Keep the final X90 scale fixed during each sweep. To validate a candidate against the original, use two scales such as `[1.0, candidate]` with fresh randomized blocks; this returns slopes without another interpolation.

In [3]:
RUN_HARDWARE = False
QUBIT = 0
EXPERIMENT = 'ramsey'  # 'readout', 'ramsey', 'ramsey_three_stage', 'pingpong'
SHOTS = 1024

RAMSEY_STAGE = 0       # 0: 10 MHz / 0.3 µs; 1: 1 MHz / 3 µs; 2: 0.1 MHz / 30 µs
CENTER_OFFSET_MHZ = 0.0
X90_SCALE = 1.0        # fixed final half pulse; also both pulses in Ramsey

GATE = 'X90'          # independent repeated pulse: 'X90' or 'X'
SCALES = [0.98, 1.0, 1.02]
COUNTS = [0, 4, 8, 12] if GATE == 'X90' else [0, 2, 4, 6]
BLOCKS = 2
SEED = 20260929       # change for an independent validation scan

READOUT_REFERENCE = _readout_references.get(QUBIT, HARNESS_ROOT / f'artifacts/x6y3/leeq-reproduction/hardware/Q{QUBIT}-readout')
OUTPUT_ROOT = HARNESS_ROOT / 'artifacts/x6y3/jupyter-experiments'
print('Hardware enabled:', RUN_HARDWARE, '| Qubit:', QUBIT, '| Experiment:', EXPERIMENT)

Hardware enabled: False | Qubit: 0 | Experiment: ramsey


In [4]:
def make_plan(center=None, stage=None):
    center = CENTER_OFFSET_MHZ if center is None else center
    if EXPERIMENT == 'readout':
        return readout_plan(QUBIT, shots=SHOTS, blocks=max(2, BLOCKS))
    if EXPERIMENT in ('ramsey', 'ramsey_three_stage'):
        stage = RAMSEY_STAGE if stage is None else stage
        return leeq_ramsey_plan(QUBIT, stage, center_offset_mhz=center,
                                x90_scale=X90_SCALE, shots=SHOTS)
    if EXPERIMENT == 'pingpong':
        return pingpong_plan(QUBIT, GATE, SCALES, COUNTS,
                             center_offset_mhz=center, x90_scale=X90_SCALE,
                             shots=SHOTS, blocks=BLOCKS, seed=SEED)
    raise ValueError('Unknown experiment')

# Offline preview: representative points independently checked against qcal.
import copy
plan = make_plan(stage=0 if EXPERIMENT == 'ramsey_three_stage' else None)
preview = copy.deepcopy(plan)
preview.points = [plan.points[i] for i in sorted({0, len(plan.points)//2, len(plan.points)-1})]
parity = verify_plan_parity(calibration, huracan, preview)
print(f'Plan: {plan.name}, {len(plan.points)} points, {plan.shots} shots/point')
print(f'Offline qcal parity passed for {parity["points_verified"]} representative points.')
print('Three-stage mode runs 3 × 60 points, stopping if a fit is rejected.' if EXPERIMENT == 'ramsey_three_stage' else '')

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Plan: leeq_ramsey, 60 points, 1024 shots/point
Offline qcal parity passed for 3 representative points.



## Acquire the selected experiment

This is the only cell that contacts the board. With hardware disabled it prints a preview message. With hardware enabled it saves each completed point, the exact original calibration and provenance. Complete raw IQ remains available even if analysis fails. No retries or calibration promotion occur automatically.

In three-stage Ramsey mode, each accepted fit supplies the next center. Q0's previous 30 µs fit was rejected; inspect the result rather than forcing a frequency update. A single-stage scan lets you choose the last reliable center yourself.

In [5]:
results = []
reports = []
if not RUN_HARDWARE:
    print('Dry run only. Set RUN_HARDWARE=True in the controls and rerun that cell to acquire.')
else:
    import socket
    socket.setdefaulttimeout(60)
    manager = ExperimentManager()
    if manager.get_default_setup() is not huracan:
        if huracan.name in manager.get_available_setup_names():
            manager.set_default_setup(huracan.name)
        else:
            manager.register_setup(huracan)
    if EXPERIMENT != 'readout':
        ref_manifest = json.loads((READOUT_REFERENCE / 'manifest.json').read_text())
        if (ref_manifest['status'] != 'complete' or ref_manifest['qubit'] != QUBIT
                or ref_manifest['calibration_sha256'] != calibration.sha256):
            raise ValueError('Use a complete readout reference matching this qubit and baseline')
        readout_report = json.loads((READOUT_REFERENCE / 'analysis.json').read_text())
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
    session_output = OUTPUT_ROOT / f'{stamp}-Q{QUBIT}-{EXPERIMENT}'
    center = CENTER_OFFSET_MHZ
    stages = range(3) if EXPERIMENT == 'ramsey_three_stage' else [None]
    for stage in stages:
        plan = make_plan(center=center, stage=stage)
        directory = session_output / (f'stage{stage}' if stage is not None else 'scan')
        result = acquire_plan(calibration, huracan, plan, directory)
        results.append(result)
        if EXPERIMENT == 'readout':
            report = analyze_readout(result)
            READOUT_REFERENCE = directory
            _readout_references[QUBIT] = directory
        elif EXPERIMENT in ('ramsey', 'ramsey_three_stage'):
            report = fit_ramsey(result, readout_report)
        else:
            report = fit_pingpong(result, readout_report)
        reports.append(report)
        (directory / 'analysis.json').write_text(json.dumps(report, indent=2) + '\n')
        print('Saved:', directory)
        display({k:v for k,v in report.items() if k not in ('mean', 'standard_error', 'curves')})
        if EXPERIMENT == 'ramsey_three_stage':
            if not report['accepted']:
                print('STOP: fit rejected. Remaining stages were not run. Inspect saved data.')
                break
            center = report['new_center_offset_mhz']
    print('Acquisition finished. No calibration values were written back.')

Dry run only. Set RUN_HARDWARE=True in the controls and rerun that cell to acquire.


## Plot and inspect

IQ projection uses nominal preparation centers and is not an independently calibrated population or fidelity. Ping-pong zero crossings require a fresh validation scan; Q0's drift can dominate the statistical fit uncertainty.

In [6]:
for result, report in zip(results, reports):
    points = result['manifest']['points']
    fig, ax = plt.subplots(figsize=(8, 4))
    if result['manifest']['experiment'] == 'readout':
        for state in (0, 1):
            mask = [p['prepared_state'] == state for p in points]
            iq = result['iq'][mask].ravel()
            ax.scatter(iq.real, iq.imag, s=2, alpha=.2, label=f'Nominal preparation {state}')
        ax.set(xlabel='I', ylabel='Q', title='Readout reference')
    elif result['manifest']['experiment'] == 'leeq_ramsey':
        t = np.array([p['delay_us'] for p in points])
        ax.errorbar(t, report['mean'], yerr=report['standard_error'], fmt='.', label='IQ projection')
        p = {k:v['value'] for k,v in report['parameters'].items()}
        xx = np.linspace(t.min(), t.max(), 800)
        ax.plot(xx, p['Amplitude']*np.exp(-xx/p['Decay'])*np.sin(2*np.pi*p['Frequency']*xx+p['Phase'])+p['Offset'], label='Fit')
        ax.set(xlabel='Delay (µs)', ylabel='IQ projection', title=f"Ramsey: accepted={report['accepted']}; R²={report['r_squared']:.3f}")
    else:
        for c in report['curves']:
            x = np.array([p['pulse_count'] for p in c['paired_points']])
            y = [p['difference'] for p in c['paired_points']]
            line, = ax.plot(x, y, '.', label=f"{c['amplitude_scale']:.6f}×; slope {c['slope']:.4f}±{c['slope_std']:.4f}")
            xx = np.array([x.min(), x.max()])
            ax.plot(xx, c['slope']*xx+c['intercept'], color=line.get_color())
        ax.axhline(0, color='gray', lw=.6)
        ax.set(xlabel='Repeated gate count', ylabel='Final-phase paired IQ projection', title='Ping-pong amplitude scan')
    ax.legend(fontsize=8); fig.tight_layout()
    fig.savefig(result['output_dir'] / 'plot.png', dpi=160)
    plt.show()

## Optional export after independent validation

Enter reviewed frequency and independent X90/X amplitude scales below. This writes a new local YAML and refuses to overwrite the input calibration. It does not configure or contact the board.

In [7]:
SAVE_VALIDATED_CANDIDATE = False
validated_settings = {}  # {0: {'frequency_offset_mhz': ..., 'X90_scale': ..., 'X_scale': ...}}
if SAVE_VALIDATED_CANDIDATE:
    if not validated_settings:
        raise ValueError('Enter independently validated settings first')
    candidate = OUTPUT_ROOT / ('candidate-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '.yaml')
    candidate.parent.mkdir(parents=True, exist_ok=True)
    save_candidate(calibration, validated_settings, candidate)
    print('Local candidate saved:', candidate)